# 03 - Model Training & MLflow Tracking

This notebook demonstrates training **all forecasting model families** on the M5 dataset:

| Family | Models |
|--------|--------|
| Statistical | SARIMAX, Prophet, ETS |
| Machine Learning | XGBoost, LightGBM (with Optuna tuning) |
| Deep Learning | LSTM, N-BEATS, TFT |
| Ensemble | WeightedEnsemble, StackingEnsemble |

Every model run is logged to **MLflow** so we can compare metrics and reproduce results.

---
## 1. Setup

In [ ]:
import sys
import os
import warnings

sys.path.insert(0, '..')
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import mlflow
import yaml

# Model classes
from src.models.statistical import SARIMAXForecaster, ProphetForecaster, ETSForecaster
from src.models.ml_models import XGBoostForecaster, LightGBMForecaster
from src.models.deep_learning import LSTMForecaster, NBEATSForecaster, TFTForecaster
from src.models.ensemble import WeightedEnsemble, StackingEnsemble

# Evaluation
from src.evaluation.metrics import rmse, mae, smape, compute_all_metrics

# Plot style
plt.rcParams.update({
    'figure.figsize': (14, 5),
    'axes.titlesize': 13,
    'axes.labelsize': 11,
    'lines.linewidth': 1.5,
})
sns.set_style('whitegrid')

SEED = 42
np.random.seed(SEED)

print('Setup complete.')

In [ ]:
# Load project config
with open('../config/config.yaml', 'r') as f:
    config = yaml.safe_load(f)

TARGET_COL = config['data']['target_col']      # 'sales'
DATE_COL = config['data']['date_col']           # 'date'
HORIZON = config['data']['forecast_horizon']    # 28

print(f'Target: {TARGET_COL}  |  Date col: {DATE_COL}  |  Horizon: {HORIZON}')

In [ ]:
# Configure MLflow
mlflow.set_tracking_uri(config['mlflow']['tracking_uri'])
mlflow.set_experiment(config['mlflow']['experiment_name'])

print(f"MLflow tracking URI : {mlflow.get_tracking_uri()}")
print(f"MLflow experiment   : {config['mlflow']['experiment_name']}")

In [ ]:
# Load featured dataset
df = pd.read_parquet('../data/processed/sales_featured.parquet')
print(f'Loaded sales_featured.parquet: {df.shape}')
df.head()

---
## 2. Data Preparation

We sample a tractable subset of 100 random item-store pairs and create a train/test split
using the last 28 days as the test set.

In [ ]:
# Build a unique series identifier if it doesn't already exist
if 'id' not in df.columns:
    if 'item_id' in df.columns and 'store_id' in df.columns:
        df['id'] = df['item_id'].astype(str) + '_' + df['store_id'].astype(str)
    else:
        # Fallback: treat the whole frame as one series
        df['id'] = 'series_0'

all_ids = df['id'].unique()
print(f'Total unique series: {len(all_ids)}')

# Sample 100 series (or fewer if the dataset is smaller)
N_SAMPLE = min(100, len(all_ids))
sampled_ids = np.random.choice(all_ids, size=N_SAMPLE, replace=False)
df_sampled = df[df['id'].isin(sampled_ids)].copy().sort_values([DATE_COL, 'id']).reset_index(drop=True)

print(f'Sampled {N_SAMPLE} series -> {len(df_sampled):,} rows')

In [ ]:
# Ensure date column is datetime
df_sampled[DATE_COL] = pd.to_datetime(df_sampled[DATE_COL])

# Train / test split: last HORIZON days = test
cutoff_date = df_sampled[DATE_COL].max() - pd.Timedelta(days=HORIZON - 1)

train_mask = df_sampled[DATE_COL] < cutoff_date
test_mask = df_sampled[DATE_COL] >= cutoff_date

df_train = df_sampled[train_mask].copy()
df_test = df_sampled[test_mask].copy()

print(f'Cutoff date : {cutoff_date.date()}')
print(f'Train shape : {df_train.shape}')
print(f'Test  shape : {df_test.shape}')

In [ ]:
# Identify feature columns (all numeric columns except target, id, date)
EXCLUDE_COLS = {TARGET_COL, DATE_COL, 'id', 'item_id', 'store_id', 'dept_id', 'cat_id', 'state_id', 'd'}
feature_cols = [
    c for c in df_train.select_dtypes(include=[np.number]).columns
    if c not in EXCLUDE_COLS
]
print(f'{len(feature_cols)} feature columns identified')
print(feature_cols[:15], '...' if len(feature_cols) > 15 else '')

In [ ]:
# Pick a single series for statistical and deep-learning demos
demo_id = sampled_ids[0]
df_demo = df_sampled[df_sampled['id'] == demo_id].sort_values(DATE_COL).reset_index(drop=True)

demo_train = df_demo[df_demo[DATE_COL] < cutoff_date].copy()
demo_test = df_demo[df_demo[DATE_COL] >= cutoff_date].copy()

print(f'Demo series: {demo_id}')
print(f'  Train: {len(demo_train)} rows  |  Test: {len(demo_test)} rows')

In [ ]:
# Helper: collect all results in a summary list
results_collector = []

def log_result(model_name, y_true, y_pred):
    """Compute metrics for a model and append to the collector."""
    metrics = compute_all_metrics(y_true, y_pred)
    metrics['model'] = model_name
    results_collector.append(metrics)
    print(f"  {model_name:25s}  RMSE={metrics['rmse']:.4f}  MAE={metrics['mae']:.4f}  SMAPE={metrics['smape']:.2f}")
    return metrics

---
## 3. Statistical Models (single series demo)

Statistical models operate on individual time series. We demonstrate SARIMAX, Prophet, and
ETS on the demo series and log every run to MLflow.

### 3.1 SARIMAX

In [ ]:
sarimax_cfg = config['models']['statistical']['sarimax']

sarimax = SARIMAXForecaster(
    order=tuple(sarimax_cfg['order']),
    seasonal_order=tuple(sarimax_cfg['seasonal_order']),
)

with mlflow.start_run(run_name='SARIMAX_demo'):
    sarimax.fit(demo_train, target_col=TARGET_COL)
    sarimax_preds = sarimax.predict(horizon=HORIZON)

    y_true_demo = demo_test[TARGET_COL].values[:len(sarimax_preds)]
    metrics_sarimax = log_result('SARIMAX', y_true_demo, sarimax_preds)

    mlflow.log_params(sarimax.get_params())
    mlflow.log_metrics({k: v for k, v in metrics_sarimax.items() if k != 'model'})
    mlflow.set_tag('model_family', 'statistical')

### 3.2 Prophet

In [ ]:
prophet_cfg = config['models']['statistical']['prophet']

prophet_model = ProphetForecaster(
    changepoint_prior_scale=prophet_cfg['changepoint_prior_scale'],
    seasonality_prior_scale=prophet_cfg['seasonality_prior_scale'],
    yearly_seasonality=prophet_cfg['yearly_seasonality'],
    weekly_seasonality=prophet_cfg['weekly_seasonality'],
)

with mlflow.start_run(run_name='Prophet_demo'):
    prophet_model.fit(demo_train, target_col=TARGET_COL, date_col=DATE_COL)
    prophet_preds = prophet_model.predict(horizon=HORIZON)

    y_true_demo = demo_test[TARGET_COL].values[:len(prophet_preds)]
    metrics_prophet = log_result('Prophet', y_true_demo, prophet_preds)

    mlflow.log_params(prophet_model.get_params())
    mlflow.log_metrics({k: v for k, v in metrics_prophet.items() if k != 'model'})
    mlflow.set_tag('model_family', 'statistical')

### 3.3 ETS (Exponential Smoothing)

In [ ]:
ets_cfg = config['models']['statistical']['ets']

ets_model = ETSForecaster(
    seasonal_periods=ets_cfg['seasonal_periods'],
    trend='add',
    seasonal='add',
)

with mlflow.start_run(run_name='ETS_demo'):
    ets_model.fit(demo_train, target_col=TARGET_COL)
    ets_preds = ets_model.predict(horizon=HORIZON)

    y_true_demo = demo_test[TARGET_COL].values[:len(ets_preds)]
    metrics_ets = log_result('ETS', y_true_demo, ets_preds)

    mlflow.log_params(ets_model.get_params())
    mlflow.log_metrics({k: v for k, v in metrics_ets.items() if k != 'model'})
    mlflow.set_tag('model_family', 'statistical')

### 3.4 Statistical Model Forecasts vs Actual

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))

# Plot last 60 days of training for context
context = demo_train.tail(60)
ax.plot(context[DATE_COL], context[TARGET_COL], color='black', label='Train (last 60d)', alpha=0.6)

# Actual test
test_dates = demo_test[DATE_COL].values[:HORIZON]
ax.plot(test_dates, y_true_demo, color='black', linewidth=2, label='Actual')

# Forecasts
ax.plot(test_dates, sarimax_preds, '--', label=f'SARIMAX (RMSE={metrics_sarimax["rmse"]:.2f})')
ax.plot(test_dates, prophet_preds, '--', label=f'Prophet (RMSE={metrics_prophet["rmse"]:.2f})')
ax.plot(test_dates, ets_preds, '--', label=f'ETS (RMSE={metrics_ets["rmse"]:.2f})')

ax.axvline(cutoff_date, color='red', linestyle=':', alpha=0.7, label='Train/Test cutoff')
ax.set_title(f'Statistical Models - Forecast vs Actual  [{demo_id}]')
ax.set_xlabel('Date')
ax.set_ylabel(TARGET_COL)
ax.legend(loc='best')
plt.tight_layout()
plt.show()

---
## 4. Machine Learning Models (full feature matrix)

XGBoost and LightGBM are trained on the full sampled panel dataset. These models
work with the tabular feature matrix across all series simultaneously.

In [ ]:
# Fill any remaining NaNs in features for ML models
df_train_ml = df_train[feature_cols + [TARGET_COL]].fillna(0).copy()
df_test_ml = df_test[feature_cols + [TARGET_COL]].fillna(0).copy()

y_test_ml = df_test_ml[TARGET_COL].values

print(f'ML training set : {df_train_ml.shape}')
print(f'ML test set     : {df_test_ml.shape}')

### 4.1 XGBoost

In [ ]:
xgb_cfg = config['models']['ml']['xgboost']

xgb_model = XGBoostForecaster(
    n_estimators=xgb_cfg['n_estimators'],
    max_depth=xgb_cfg['max_depth'],
    learning_rate=xgb_cfg['learning_rate'],
    subsample=xgb_cfg['subsample'],
    colsample_bytree=xgb_cfg['colsample_bytree'],
    early_stopping_rounds=xgb_cfg['early_stopping_rounds'],
    val_size=HORIZON,
    random_state=SEED,
)

with mlflow.start_run(run_name='XGBoost'):
    xgb_model.fit(df_train_ml, target_col=TARGET_COL, feature_cols=feature_cols)
    xgb_preds = xgb_model.predict(df_test_ml)

    metrics_xgb = log_result('XGBoost', y_test_ml, xgb_preds)

    mlflow.log_params(xgb_model.get_params())
    mlflow.log_metrics({k: v for k, v in metrics_xgb.items() if k != 'model'})
    mlflow.set_tag('model_family', 'ml')

### 4.2 LightGBM

In [ ]:
lgb_cfg = config['models']['ml']['lightgbm']

lgb_model = LightGBMForecaster(
    n_estimators=lgb_cfg['n_estimators'],
    max_depth=lgb_cfg['max_depth'],
    learning_rate=lgb_cfg['learning_rate'],
    subsample=lgb_cfg['subsample'],
    colsample_bytree=lgb_cfg['colsample_bytree'],
    early_stopping_rounds=lgb_cfg['early_stopping_rounds'],
    val_size=HORIZON,
    random_state=SEED,
)

with mlflow.start_run(run_name='LightGBM'):
    lgb_model.fit(df_train_ml, target_col=TARGET_COL, feature_cols=feature_cols)
    lgb_preds = lgb_model.predict(df_test_ml)

    metrics_lgb = log_result('LightGBM', y_test_ml, lgb_preds)

    mlflow.log_params(lgb_model.get_params())
    mlflow.log_metrics({k: v for k, v in metrics_lgb.items() if k != 'model'})
    mlflow.set_tag('model_family', 'ml')

### 4.3 Optuna Hyperparameter Tuning (small demo)

In [ ]:
# Tune XGBoost with a small number of trials for demo purposes
N_TRIALS = 10

xgb_tuned = XGBoostForecaster(val_size=HORIZON, random_state=SEED)

with mlflow.start_run(run_name='XGBoost_Optuna'):
    best_params_xgb = xgb_tuned.tune(
        df_train_ml,
        target_col=TARGET_COL,
        feature_cols=feature_cols,
        n_trials=N_TRIALS,
    )
    xgb_tuned_preds = xgb_tuned.predict(df_test_ml)

    metrics_xgb_tuned = log_result('XGBoost (tuned)', y_test_ml, xgb_tuned_preds)

    mlflow.log_params({f'tuned_{k}': v for k, v in best_params_xgb.items()})
    mlflow.log_metrics({k: v for k, v in metrics_xgb_tuned.items() if k != 'model'})
    mlflow.log_param('n_trials', N_TRIALS)
    mlflow.set_tag('model_family', 'ml')
    mlflow.set_tag('tuned', 'optuna')

print(f'\nBest XGBoost params from Optuna: {best_params_xgb}')

In [ ]:
# Tune LightGBM similarly
lgb_tuned = LightGBMForecaster(val_size=HORIZON, random_state=SEED)

with mlflow.start_run(run_name='LightGBM_Optuna'):
    best_params_lgb = lgb_tuned.tune(
        df_train_ml,
        target_col=TARGET_COL,
        feature_cols=feature_cols,
        n_trials=N_TRIALS,
    )
    lgb_tuned_preds = lgb_tuned.predict(df_test_ml)

    metrics_lgb_tuned = log_result('LightGBM (tuned)', y_test_ml, lgb_tuned_preds)

    mlflow.log_params({f'tuned_{k}': v for k, v in best_params_lgb.items()})
    mlflow.log_metrics({k: v for k, v in metrics_lgb_tuned.items() if k != 'model'})
    mlflow.log_param('n_trials', N_TRIALS)
    mlflow.set_tag('model_family', 'ml')
    mlflow.set_tag('tuned', 'optuna')

print(f'\nBest LightGBM params from Optuna: {best_params_lgb}')

### 4.4 Feature Importance

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# XGBoost feature importance
xgb_imp = xgb_model.get_feature_importance(top_n=15)
axes[0].barh(xgb_imp['feature'], xgb_imp['importance'], color='steelblue')
axes[0].set_title('XGBoost - Top 15 Features')
axes[0].set_xlabel('Importance')
axes[0].invert_yaxis()

# LightGBM feature importance
lgb_imp = lgb_model.get_feature_importance(top_n=15)
axes[1].barh(lgb_imp['feature'], lgb_imp['importance'], color='darkorange')
axes[1].set_title('LightGBM - Top 15 Features')
axes[1].set_xlabel('Importance')
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

---
## 5. Deep Learning Models (sampled series)

Deep learning models are demonstrated on the demo series (LSTM) or a small panel
subset (N-BEATS, TFT).

### 5.1 LSTM

In [ ]:
lstm_cfg = config['models']['deep_learning']['lstm']

lstm_model = LSTMForecaster(
    hidden_size=lstm_cfg['hidden_size'],
    num_layers=lstm_cfg['num_layers'],
    dropout=lstm_cfg['dropout'],
    seq_length=HORIZON,
    batch_size=lstm_cfg['batch_size'],
    epochs=lstm_cfg['epochs'],
    lr=lstm_cfg['learning_rate'],
    random_state=SEED,
)

with mlflow.start_run(run_name='LSTM_demo'):
    lstm_model.fit(demo_train, target_col=TARGET_COL)

    # For LSTM prediction, provide a context window + horizon rows
    lstm_input = pd.concat([demo_train.tail(HORIZON), demo_test], ignore_index=True)
    lstm_preds = lstm_model.predict(lstm_input, horizon=HORIZON)

    y_true_demo = demo_test[TARGET_COL].values[:len(lstm_preds)]
    metrics_lstm = log_result('LSTM', y_true_demo, lstm_preds)

    mlflow.log_params(lstm_model.get_params())
    mlflow.log_metrics({k: v for k, v in metrics_lstm.items() if k != 'model'})
    mlflow.set_tag('model_family', 'deep_learning')

### 5.2 N-BEATS

In [ ]:
nbeats_cfg = config['models']['deep_learning']['nbeats']

nbeats_model = NBEATSForecaster(
    input_size=nbeats_cfg['input_size'],
    h=nbeats_cfg['h'],
    max_steps=nbeats_cfg['max_steps'],
    random_state=SEED,
)

with mlflow.start_run(run_name='NBEATS_demo'):
    nbeats_model.fit(
        demo_train,
        target_col=TARGET_COL,
        date_col=DATE_COL,
        unique_id=demo_id,
    )
    nbeats_preds = nbeats_model.predict()

    y_true_demo = demo_test[TARGET_COL].values[:len(nbeats_preds)]
    metrics_nbeats = log_result('N-BEATS', y_true_demo, nbeats_preds)

    mlflow.log_params(nbeats_model.get_params())
    mlflow.log_metrics({k: v for k, v in metrics_nbeats.items() if k != 'model'})
    mlflow.set_tag('model_family', 'deep_learning')

### 5.3 Temporal Fusion Transformer (TFT)

In [ ]:
tft_cfg = config['models']['deep_learning']['tft']

# Prepare TFT-specific data: needs time_idx and group_id columns
df_tft = df_demo.copy()
df_tft['time_idx'] = np.arange(len(df_tft))
df_tft['group_id'] = demo_id

tft_train = df_tft[df_tft[DATE_COL] < cutoff_date].copy()
tft_test = df_tft[df_tft[DATE_COL] >= cutoff_date].copy()

# Identify known real-valued covariates from calendar features
calendar_features = [c for c in config['features']['calendar_features'] if c in df_tft.columns]

tft_model = TFTForecaster(
    hidden_size=tft_cfg['hidden_size'],
    attention_head_size=tft_cfg['attention_head_size'],
    dropout=tft_cfg['dropout'],
    max_prediction_length=HORIZON,
    max_encoder_length=60,
    max_epochs=tft_cfg['max_epochs'],
    batch_size=tft_cfg['batch_size'],
    lr=tft_cfg['learning_rate'],
    time_varying_known_reals=calendar_features,
    random_state=SEED,
)

with mlflow.start_run(run_name='TFT_demo'):
    tft_model.fit(
        tft_train,
        target_col=TARGET_COL,
        time_idx_col='time_idx',
        group_col='group_id',
    )

    # Predict on the combined data (TFT uses the dataset format internally)
    tft_preds = tft_model.predict(
        df_tft,
        time_idx_col='time_idx',
        group_col='group_id',
    )

    y_true_tft = tft_test[TARGET_COL].values[:len(tft_preds)]
    metrics_tft = log_result('TFT', y_true_tft, tft_preds)

    mlflow.log_params(tft_model.get_params())
    mlflow.log_metrics({k: v for k, v in metrics_tft.items() if k != 'model'})
    mlflow.set_tag('model_family', 'deep_learning')

---
## 6. Ensemble Models

We combine the ML models (XGBoost + LightGBM) into ensembles.

### 6.1 Weighted Ensemble

In [ ]:
weighted_ensemble = WeightedEnsemble(
    forecasters=[
        ('XGBoost', XGBoostForecaster(
            n_estimators=xgb_cfg['n_estimators'],
            max_depth=xgb_cfg['max_depth'],
            learning_rate=xgb_cfg['learning_rate'],
            val_size=HORIZON,
            random_state=SEED,
        )),
        ('LightGBM', LightGBMForecaster(
            n_estimators=lgb_cfg['n_estimators'],
            max_depth=lgb_cfg['max_depth'],
            learning_rate=lgb_cfg['learning_rate'],
            val_size=HORIZON,
            random_state=SEED,
        )),
    ],
    val_size=HORIZON,
)

with mlflow.start_run(run_name='WeightedEnsemble'):
    weighted_ensemble.fit(df_train_ml, target_col=TARGET_COL, feature_cols=feature_cols)
    we_preds = weighted_ensemble.predict(df_test_ml)

    metrics_we = log_result('WeightedEnsemble', y_test_ml, we_preds)

    mlflow.log_params(weighted_ensemble.get_params())
    mlflow.log_metrics({k: v for k, v in metrics_we.items() if k != 'model'})
    mlflow.set_tag('model_family', 'ensemble')

# Display optimised weights
print('\nOptimal ensemble weights:')
weighted_ensemble.get_model_weights()

### 6.2 Stacking Ensemble

In [ ]:
stacking_ensemble = StackingEnsemble(
    forecasters=[
        ('XGBoost', XGBoostForecaster(
            n_estimators=xgb_cfg['n_estimators'],
            max_depth=xgb_cfg['max_depth'],
            learning_rate=xgb_cfg['learning_rate'],
            val_size=HORIZON,
            random_state=SEED,
        )),
        ('LightGBM', LightGBMForecaster(
            n_estimators=lgb_cfg['n_estimators'],
            max_depth=lgb_cfg['max_depth'],
            learning_rate=lgb_cfg['learning_rate'],
            val_size=HORIZON,
            random_state=SEED,
        )),
    ],
    meta_learner=config['models']['ensemble']['meta_learner'],
    n_splits=config['models']['ensemble']['cv_folds'],
    test_size=HORIZON,
)

with mlflow.start_run(run_name='StackingEnsemble'):
    stacking_ensemble.fit(df_train_ml, target_col=TARGET_COL, feature_cols=feature_cols)
    se_preds = stacking_ensemble.predict(df_test_ml)

    metrics_se = log_result('StackingEnsemble', y_test_ml, se_preds)

    mlflow.log_params(stacking_ensemble.get_params())
    mlflow.log_metrics({k: v for k, v in metrics_se.items() if k != 'model'})
    mlflow.set_tag('model_family', 'ensemble')

# Display meta-learner coefficients
print('\nMeta-learner model weights:')
stacking_ensemble.get_model_weights()

---
## 7. Model Comparison

Summary table of all trained models ranked by RMSE.

In [ ]:
results_df = pd.DataFrame(results_collector)

# Reorder columns
col_order = ['model', 'rmse', 'mae', 'smape']
extra_cols = [c for c in results_df.columns if c not in col_order]
results_df = results_df[col_order + extra_cols]

results_df = results_df.sort_values('rmse').reset_index(drop=True)
results_df.style.format({
    'rmse': '{:.4f}',
    'mae': '{:.4f}',
    'smape': '{:.2f}',
}).background_gradient(subset=['rmse'], cmap='RdYlGn_r')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for ax, metric in zip(axes, ['rmse', 'mae', 'smape']):
    sorted_df = results_df.sort_values(metric)
    colors = plt.cm.RdYlGn_r(np.linspace(0.2, 0.8, len(sorted_df)))
    ax.barh(sorted_df['model'], sorted_df[metric], color=colors)
    ax.set_title(metric.upper())
    ax.set_xlabel(metric.upper())
    ax.invert_yaxis()

plt.suptitle('Model Comparison Across Metrics', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
best_model = results_df.iloc[0]
print(f"Best model by RMSE: {best_model['model']}")
print(f"  RMSE  = {best_model['rmse']:.4f}")
print(f"  MAE   = {best_model['mae']:.4f}")
print(f"  SMAPE = {best_model['smape']:.2f}")
print()
print('All MLflow runs logged. Use `mlflow ui` to explore the experiment.')